# Audit a Lean file that compiles

*Adversarial Mathematics From First Principles*, [chapter 26](https://programmer.ie/books/math/26-chapter/) · [worked example](https://programmer.ie/examples/math/26-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

A file that compiles has shown less than it seems. The chapter's audit asks:
what does it state, what does each declaration rest on, is it bound to the
claimed statement, and does the statement mean the claim? This core-Lean file
is the kind of thing that might arrive with a paper: it compiles, with an
axiom of its own and a gap.

## Run it

Colab has no Lean, so the first cell downloads the official Lean 4.35.0 release (core only, about 600 MB, a minute or two). The second writes the file and the third checks it. The two warnings are part of the audit's evidence.

In [1]:
!apt-get -qq install -y zstd > /dev/null
!test -x lean-4.35.0-linux/bin/lean || curl -sSfL https://github.com/leanprover/lean4/releases/download/v4.35.0/lean-4.35.0-linux.tar.zst | tar --zstd -x
!lean-4.35.0-linux/bin/lean --version

Lean (version 4.35.0, x86_64-unknown-linux-gnu, commit eb6cafb7290e8bd5f2937522d8d804f58f02bafc, Release)


In [2]:
%%writefile audit.lean
-- Auditing someone else's proof: the questions to ask of a file that compiles (core Lean 4).
-- Imagine this arrived attached to a paper.

axiom helpful_fact : ∀ n : Nat, n * n ≥ n + 1                  -- 4. an axiom of its own (false at n = 1)

theorem lemma_ok (n : Nat) : n ≤ n * n + n := by omega
theorem lemma_gap (n : Nat) : n * n ≥ n := sorry                  -- a gap
theorem main_result (n : Nat) (h : 1 < n) : n < n * n := by
  have := helpful_fact n
  omega

-- 3. What does it state? Read the statements, not the names.
#check @main_result
-- 4. What does each declaration rest on?
#print axioms lemma_ok
#print axioms lemma_gap
#print axioms main_result
-- 5. Is it bound to the claimed statement, "every n > 1 is less than its square"?
example : ∀ n : Nat, 1 < n → n < n * n := main_result
-- The axiom is false: the audit can refute it independently.
example : ¬ (∀ n : Nat, n * n ≥ n + 1) := fun h => absurd (h 1) (by decide)

Writing audit.lean


In [3]:
!lean-4.35.0-linux/bin/lean audit.lean

audit.lean:7:8: warning: declaration uses `sorry`
audit.lean:8:31: warning: Variable name `h` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _h

Note: This linter can be disabled with `set_option linter.unusedVariables false`
main_result : ∀ (n : Nat), 1 < n → n < n * n
'lemma_ok' depends on axioms: [propext, Quot.sound]
'lemma_gap' depends on axioms: [sorryAx]
'main_result' depends on axioms: [helpful_fact, propext, Quot.sound]


## What the output shows

`main_result` states exactly the claim and binds to it, but `#print axioms`
shows it rests on `helpful_fact`, an axiom the file declares. Lean's own
warning shows the hypothesis `h` is not even used: the result follows from the
axiom alone. And the axiom is false at n = 1, which the last line proves. So
the file proves nothing about the claim. `lemma_gap` rests on `sorryAx`.

## Not shown

- A constructed example; Part I, chapter 5 applies the same audit to a real published artifact.
- Finding the axiom is mechanical; deciding that it is false, or acceptable, is not.

## Challenge it

Apply the chapter's audit checklist to this file. For each question, give the
evidence from Lean's output. Then rewrite `main_result` without the axiom, prove
it, and say what `#print axioms` should report afterwards.